# Uniform Manifold Approximation and Projection (UMAP)

O UMAP é da mesma família do t-SNE: também tenta manter juntos, no mapa 2D, os pontos que eram vizinhos no espaço original. A diferença está no caminho, que tem dois passos bem concretos. Primeiro ele **desenha uma rede de vizinhos**, ligando cada ponto aos mais próximos, como uma rede de amizades. Depois **arruma essa rede no plano**, tratando cada ligação como uma mola e deixando o sistema se acomodar. Na prática, o UMAP costuma ser mais rápido que o t-SNE, preserva um pouco melhor a posição relativa dos grupos e, ao contrário dele, consegue projetar dados novos depois de treinado.

Neste notebook usamos uma folha enrolada em espiral para entender o problema, construímos a rede de vizinhos e o layout passo a passo, implementamos a classe do zero e terminamos aplicando a biblioteca a dígitos manuscritos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap
from sklearn.datasets import load_digits, make_swiss_roll
from sklearn.manifold import SpectralEmbedding
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')

A biblioteca de referência não faz parte do Scikit-Learn e precisa ser instalada à parte com `pip install umap-learn`. O pacote se chama `umap-learn` na instalação e `umap` no `import`.

## Dataset: Swiss Roll

O *swiss roll* é uma folha enrolada em espiral no espaço 3D, e a cor de cada ponto indica sua posição **ao longo da folha**. Um bom mapa 2D deveria desenrolá-la, com as cores variando suavemente de uma ponta à outra.

A dificuldade aparece em dois pontos de voltas vizinhas da espiral, marcados com estrelas: em linha reta eles estão **perto**, mas andando sobre a folha estão **muito longe**. Qualquer método que confie na distância em linha reta entre pontos distantes vai ter dificuldade em desenrolá-la.

In [ ]:
X, color = make_swiss_roll(n_samples=1500, noise=0.05, random_state=42)

# um ponto no meio da folha e o mais próximo dele em linha reta
# entre os que estão pelo menos uma volta à frente na espiral
i = np.argmin(np.abs(color - 8))
ahead = np.where(color > color[i] + 5)[0]
j = ahead[np.argmin(np.linalg.norm(X[ahead] - X[i], axis=1))]
print(f"Distância em linha reta entre as estrelas: {np.linalg.norm(X[i] - X[j]):.1f}")

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection='3d')
ax.scatter(*X.T, c=color, cmap='Spectral', s=6)
ax.scatter(*X[[i, j]].T, marker='*', s=400, color='black')
ax.plot(*X[[i, j]].T, color='black', lw=2)
ax.view_init(elev=10, azim=-75)
ax.set_title('Swiss roll em 3D')
plt.show()

## Fundamentação Matemática

### A Rede de Vizinhos

A ideia central é que **distâncias pequenas são confiáveis e distâncias grandes não**. Localmente a folha é quase plana, e a distância em linha reta entre pontos muito próximos é praticamente a distância sobre a folha. O UMAP então só confia na vizinhança imediata: liga cada ponto aos seus $k$ vizinhos mais próximos, o parâmetro `n_neighbors`, e ignora o resto.

In [ ]:
k = 10
_, neighbors = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection='3d')
for a in range(len(X)):
    for b in neighbors[a, 1:]:                  # neighbors[a, 0] é o próprio ponto
        ax.plot(*X[[a, b]].T, color='gray', lw=0.3, alpha=0.5)
ax.scatter(*X.T, c=color, cmap='Spectral', s=6)
ax.view_init(elev=10, azim=-75)
ax.set_title(f'Rede de vizinhos (k = {k})')
plt.show()

As ligações seguem a folha e não atravessam de uma volta para outra. Andando pela rede, a única forma de ir de uma estrela à outra é percorrer a espiral, ou seja, **a rede captura a forma da folha**.

Nem toda ligação tem a mesma força. O peso da ligação do ponto $i$ para o vizinho $j$ é

$$ w_{i \to j} = \exp\left(-\frac{d(x_i, x_j) - \rho_i}{\sigma_i}\right) $$

em que $\rho_i$ é a distância até o vizinho mais próximo. Subtraí-la garante que todo ponto tenha pelo menos uma ligação com peso 1, de modo que ninguém fica isolado, nem em regiões esparsas. Já $\sigma_i$ controla a rapidez com que o peso cai para os vizinhos mais distantes e, como no t-SNE, é ajustado por **busca binária** para cada ponto, agora de forma que a soma dos pesos seja sempre $\log_2 k$.

In [ ]:
def find_sigma(distances, rho, target):
    # Busca binária do sigma que faz a soma dos pesos valer target
    lo, hi = 1e-3, 1e3
    for _ in range(50):
        sigma = (lo + hi) / 2
        if np.sum(np.exp(-(distances - rho) / sigma)) > target:
            hi = sigma      # pesos altos demais: diminuir sigma
        else:
            lo = sigma      # pesos baixos demais: aumentar sigma
    return sigma

Comparando um ponto cujos vizinhos estão próximos, numa região densa, com um ponto cujos vizinhos estão longe, numa região esparsa:

In [ ]:
k = 15
distances, _ = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X)
distances = distances[:, 1:]                    # descarta o próprio ponto

dense, sparse = np.argmin(distances[:, -1]), np.argmax(distances[:, -1])

plt.figure(figsize=(8, 4))
for point, name in [(dense, 'ponto em região densa'), (sparse, 'ponto em região esparsa')]:
    rho = distances[point, 0]
    sigma = find_sigma(distances[point], rho, np.log2(k))
    weights = np.exp(-(distances[point] - rho) / sigma)
    plt.plot(distances[point], weights, 'o-', label=f'{name} (σ = {sigma:.2f})')

plt.xlabel(f'distância até cada um dos {k} vizinhos')
plt.ylabel('peso da ligação')
plt.legend()
plt.show()

Nos dois casos o vizinho mais próximo tem peso 1 e os pesos caem de forma parecida, mesmo com as distâncias em escalas bem diferentes: o UMAP **normaliza a densidade local**.

Como $j$ pode estar entre os vizinhos de $i$ sem que o contrário aconteça, $w_{i \to j}$ e $w_{j \to i}$ podem ser diferentes. Os dois pesos são combinados como a probabilidade de pelo menos uma das ligações existir:

$$ p_{ij} = w_{i \to j} + w_{j \to i} - w_{i \to j}\, w_{j \to i} $$

### O Layout em Baixa Dimensão

Agora os pontos precisam ser posicionados em 2D de forma que a rede fique bem representada. Cada ligação funciona como uma **mola** que puxa os dois pontos um para o outro, com as ligações de maior peso sendo usadas mais vezes. Ao mesmo tempo, cada ponto é **repelido** por alguns pontos sorteados ao acaso: num conjunto grande, um ponto sorteado quase certamente não é vizinho, então isso funciona como uma repulsão barata. Esse truque se chama **amostragem negativa** e é o que torna o UMAP rápido, porque ele nunca compara todos os pares.

No mapa, a similaridade entre dois pontos a distância $d$ é medida pela curva

$$ q(d) = \frac{1}{1 + a\, d^{2b}} $$

que, com $a = b = 1$, é exatamente a t de Student do t-SNE. Os valores de $a$ e $b$ são ajustados a partir do hiperparâmetro `min_dist`, que diz o quão perto os pontos podem ficar no mapa; com o padrão `min_dist = 0.1`, $a \approx 1{,}58$ e $b \approx 0{,}90$.

In [ ]:
d = np.linspace(0, 4, 200)

plt.figure(figsize=(8, 4))
plt.plot(d, 1 / (1 + d ** 2), label='t de Student (t-SNE)', lw=2)
plt.plot(d, 1 / (1 + 1.58 * d ** (2 * 0.90)), label='UMAP (min_dist = 0.1)', lw=2)
plt.plot(d, 1 / (1 + 0.58 * d ** (2 * 1.33)), label='UMAP (min_dist = 0.5)', lw=2)
plt.xlabel('distância no mapa')
plt.ylabel('similaridade $q(d)$')
plt.legend()
plt.show()

Com `min_dist` maior a curva fica achatada perto de zero: pontos a distâncias pequenas são todos considerados igualmente próximos, não há motivo para empilhá-los, e o mapa fica mais espalhado.

Molas e ímãs só ajustam o desenho localmente: partindo de posições aleatórias, cada pedaço da folha se forma, mas os pedaços ficam fora de ordem. Por isso o UMAP começa de um **desenho aproximado da rede**, que já traz a estrutura global, obtido com o `SpectralEmbedding` do Scikit-Learn, que usamos pronto.

## Implementação

A classe junta os dois passos. A cada época, a taxa de aprendizado `alpha` diminui linearmente até zero, para que os pontos se acomodem no final, e a cada 10 épocas guardamos uma cópia do mapa em `history`. Usamos `np.add.at(Y, i, step)` em vez de `Y[i] += step` porque um mesmo ponto pode aparecer várias vezes em `i`, e só o `np.add.at` soma todas as contribuições.

Os coeficientes `coef` são as derivadas da função de custo do UMAP, uma entropia cruzada entre $p_{ij}$ e $q_{ij}$. O importante é o sinal: o da atração é negativo e puxa $y_i$ na direção de $y_j$, o da repulsão é positivo e empurra para longe. O `np.clip` limita cada passo para que nenhum ponto seja arremessado para fora do mapa.

In [ ]:
class UMAP:
    def __init__(self, n_neighbors=15, n_components=2, n_epochs=500, learning_rate=1.0,
                 n_negative=5, random_state=None):
        self.n_neighbors = n_neighbors
        self.n_components = n_components
        self.n_epochs = n_epochs
        self.learning_rate = learning_rate
        self.n_negative = n_negative           # pontos sorteados para repulsão, por ligação
        self.random_state = random_state
        self.a, self.b = 1.58, 0.90            # curva q(d) para min_dist = 0.1

    def _build_graph(self, X):
        # rede de vizinhos com pesos
        n, k = len(X), self.n_neighbors
        distances, indices = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X)
        distances, indices = distances[:, 1:], indices[:, 1:]

        W = np.zeros((n, n))
        for i in range(n):
            rho = distances[i, 0]
            sigma = find_sigma(distances[i], rho, np.log2(k))
            W[i, indices[i]] = np.exp(-(distances[i] - rho) / sigma)

        return W + W.T - W * W.T               # pelo menos uma das duas ligações

    def fit_transform(self, X):
        n = len(X)
        P = self._build_graph(X)
        rng = np.random.default_rng(self.random_state)

        # ponto de partida: desenho aproximado da rede
        Y = SpectralEmbedding(self.n_components, affinity='precomputed',
                              random_state=self.random_state).fit_transform(P)
        Y = 10 * Y / np.abs(Y).max()

        heads, tails = np.nonzero(P)           # lista de ligações (i, j)
        weights = P[heads, tails]
        a, b = self.a, self.b
        self.history = [Y.copy()]

        for epoch in range(self.n_epochs):
            alpha = self.learning_rate * (1 - epoch / self.n_epochs)

            # atração: cada ligação é usada com probabilidade igual ao seu peso
            used = rng.random(len(weights)) < weights
            i, j = heads[used], tails[used]
            diff = Y[i] - Y[j]
            d2 = np.sum(diff ** 2, axis=1) + 1e-6
            coef = -2 * a * b * d2 ** (b - 1) / (1 + a * d2 ** b)
            np.add.at(Y, i, alpha * np.clip(coef[:, None] * diff, -4, 4))

            # repulsão: cada ponto se afasta de n_negative pontos sorteados
            i = np.repeat(i, self.n_negative)
            j = rng.integers(0, n, len(i))
            diff = Y[i] - Y[j]
            d2 = np.sum(diff ** 2, axis=1)
            coef = 2 * b / ((0.001 + d2) * (1 + a * d2 ** b))
            np.add.at(Y, i, alpha * np.clip(coef[:, None] * diff, -4, 4))

            if (epoch + 1) % 10 == 0:
                self.history.append(Y.copy())

        return Y

In [ ]:
model = UMAP(n_neighbors=15, random_state=42)
Y = model.fit_transform(X)

plt.figure(figsize=(7, 5))
plt.scatter(Y[:, 0], Y[:, 1], c=color, cmap='Spectral', s=6)
plt.scatter(*Y[[i, j]].T, marker='*', s=400, color='black')
plt.title('UMAP (nossa implementação)')
plt.show()

A folha foi desenrolada: as cores variam suavemente de uma ponta à outra, e as duas estrelas, que estavam próximas em 3D, agora estão bem separadas, cada uma no seu trecho da faixa. O resultado fica curvado e com alguma textura, mas o que importa é a **ordem** ao longo da faixa. Pelas cópias guardadas em `history`, podemos ver como o mapa evolui.

In [ ]:
epochs = [0, 20, 50, 150, 500]

fig, axes = plt.subplots(1, len(epochs), figsize=(20, 3.8))
for ax, epoch in zip(axes, epochs):
    Y_epoch = model.history[epoch // 10]
    ax.scatter(Y_epoch[:, 0], Y_epoch[:, 1], c=color, cmap='Spectral', s=4)
    ax.set(title=f'época {epoch}', xticks=[], yticks=[])
plt.tight_layout()
plt.show()

Na época 0, o desenho espectral já põe as cores na ordem certa, mas com tudo espremido numa curva. Nas primeiras épocas a taxa de aprendizado é alta e a repulsão espalha os pontos; nas finais os passos ficam pequenos e as molas acomodam cada ponto perto dos seus vizinhos.

## O UMAP da Biblioteca

A implementação de referência segue os mesmos dois passos, com vizinhos aproximados e código compilado que a tornam muito mais rápida. Os dois hiperparâmetros principais têm papéis distintos. O `n_neighbors` (padrão 15) decide **o que** a rede enxerga: com valores muito pequenos a rede se quebra em pedaços e o mapa mostra fragmentos que não existem nos dados, e com valores muito grandes ela passa a ligar regiões distantes, como voltas diferentes da espiral. O `min_dist` (padrão 0.1) decide apenas **como** o resultado é desenhado: não muda quais pontos ficam juntos, só o quão compactos os grupos aparecem. Além deles, `n_components` define a dimensão de saída e, ao contrário do t-SNE, valores maiores que 2 também funcionam bem; `metric` escolhe a distância no espaço original, como `'cosine'` ou `'manhattan'`; e `random_state` fixa a semente.

Aplicamos o UMAP ao dataset de dígitos manuscritos, com 1797 imagens de 8×8 pixels, ou seja, 64 dimensões.

In [ ]:
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)
y_digits = digits.target

Z_digits = umap.UMAP(random_state=42).fit_transform(X_digits)

plt.figure(figsize=(8, 6.5))
plt.scatter(Z_digits[:, 0], Z_digits[:, 1], c=y_digits, cmap='tab10', s=6, alpha=0.7)
for digit in range(10):
    # escreve o dígito no centro do seu grupo
    center = np.median(Z_digits[y_digits == digit], axis=0)
    plt.text(*center, str(digit), fontsize=16, weight='bold', ha='center', va='center',
             bbox=dict(boxstyle='circle', facecolor='white', alpha=0.8))
plt.xticks([])
plt.yticks([])
plt.title('UMAP nos dígitos manuscritos')
plt.show()

A maioria dos dígitos forma grupos compactos e bem separados, e o UMAP não usou os rótulos para isso, apenas as distâncias entre as imagens. Os grupos que ficam lado a lado, como o 1 e o 8, e os pontos fora do lugar, como alguns 1 junto do 2, costumam ser dígitos de traçado parecido ou escritos de forma ambígua.

### Projetando Dados Novos

Esta é a grande vantagem prática do UMAP sobre o t-SNE: depois de treinado, ele tem um método `transform`. Para um ponto novo, ele encontra os vizinhos entre os dados de treino e o posiciona perto deles, sem refazer a otimização.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits, test_size=0.3, random_state=0)

mapper = umap.UMAP(random_state=42).fit(X_train)
Z_train = mapper.embedding_
Z_test = mapper.transform(X_test)

plt.figure(figsize=(8, 6.5))
plt.scatter(Z_train[:, 0], Z_train[:, 1], c='lightgray', s=8, label='treino')
plt.scatter(Z_test[:, 0], Z_test[:, 1], c=y_test, cmap='tab10', s=12, label='teste')
plt.legend()
plt.title('Dígitos de teste projetados no mapa de treino')
plt.show()

Os dígitos de teste, que o UMAP não viu no treino, caem nos grupos certos. Isso permite usá-lo também como etapa de pré-processamento, por exemplo antes de um algoritmo de clusterização. Vale lembrar a limitação do mecanismo: um ponto que não se pareça com nada do treino será colocado em algum lugar plausível, sem nenhum sinal de que está fora da distribuição.

## Custo e Limitações

A construção da rede custa $O(n \log n)$ com busca aproximada de vizinhos, e cada época da otimização custa $O(n)$, porque o número de ligações e de pontos sorteados é proporcional a $n$, e não a $n^2$. É isso que permite ao UMAP escalar para conjuntos que inviabilizam o t-SNE. Nossa implementação guarda uma matriz $n \times n$ e só serve para alguns milhares de pontos; a biblioteca usa estruturas esparsas.

Quanto à interpretação, valem as mesmas ressalvas do t-SNE: os eixos não significam nada, e tamanhos e distâncias no mapa não são confiáveis, porque a normalização por $\rho_i$ e $\sigma_i$ apaga deliberadamente as diferenças de densidade. O resultado depende fortemente de `n_neighbors`, e com valores pequenos a rede se fragmenta e o mapa desenha grupos que não existem. Por fim, o método assume que os dados estão sobre uma superfície de baixa dimensão; se não estiverem, o mapa continua sendo desenhado e continua parecendo convincente.

A tabela resume os três métodos de redução de dimensionalidade vistos até aqui.

| | PCA | t-SNE | UMAP |
|---|---|---|---|
| Tipo | linear | não linear | não linear |
| Preserva | variância (estrutura global) | vizinhanças (estrutura local) | vizinhanças, com mais estrutura global |
| Velocidade | muito rápido | lento | rápido |
| Projeta dados novos | sim | não | sim |
| Eixos interpretáveis | sim | não | não |
| Uso típico | compressão e pré-processamento | visualização | visualização e pré-processamento |

## Exercícios

### Exercício 1: O Efeito de rho

Modifique a classe `UMAP` para não subtrair $\rho_i$ das distâncias, usando $w_{i \to j} = \exp(-d(x_i, x_j) / \sigma_i)$. Compare os mapas do swiss roll com e sem $\rho_i$ e repita com dois grupos de densidades bem diferentes, gerados com `make_blobs` e `cluster_std=[0.5, 5.0]`. O que acontece com os pontos do grupo esparso, e que papel o $\rho_i$ cumpre?

In [ ]:
# Seu código aqui

### Exercício 2: Sem Repulsão

Rode a nossa implementação no swiss roll com `n_negative` igual a 0, 1, 5 e 20 e plote os quatro mapas, comparando também os limites dos eixos. O que acontece quando a repulsão é removida por completo, e por quê?

In [ ]:
# Seu código aqui

### Exercício 3: UMAP antes de Clusterizar

Use `umap.UMAP(n_components=10)` para reduzir os dígitos de 64 para 10 dimensões e aplique o K-Means com 10 clusters. Compare o *Adjusted Rand Index* com o obtido aplicando o K-Means diretamente às 64 dimensões e após reduzir para 10 dimensões com PCA. Qual redução ajuda mais o K-Means?

In [ ]:
# Seu código aqui